# PP1 V3: train and compare 16 models (Colab CPU only)
Phase 2B implementation. No training has run during preparation. This is within-generator synthetic demo-label imitation, not clinical severity accuracy. No new data will be generated. TS/RS are proxies; BS has no amplitude channel; MPS is a demo index, never Parkinson probability.

Use the official VS Code Google Colab extension and select CPU / no hardware accelerator. Read training/README.md. Upload the separate PP1_V3_Training_Code.zip to My Drive. The existing verified dataset stays read-only at its current Drive backup.


## 1. Verify Colab CPU and install dependencies
Run this cell only in a Colab kernel. These packages are never installed on Windows by the project. If installation requests a restart, restart the Colab kernel before proceeding.

In [1]:
from pathlib import Path
import os, sys
import google.colab
assert Path('/content').is_dir() and os.environ.get('COLAB_RELEASE_TAG'), 'Select a Colab kernel'
assert not Path('/dev/nvidia0').exists() and not os.environ.get('COLAB_TPU_ADDR'), 'Select CPU / accelerator None'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
sys.dont_write_bytecode = True
%pip install -q numpy==2.2.6 scipy==1.15.3 scikit-learn==1.6.1 xgboost==3.0.5 catboost==1.2.8 joblib==1.4.2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 MB 5.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.5/351.5 MB 3.8 MB/s eta 0:00:00:00:0100:01


## 2. Mount Drive and restore training code; open existing dataset read-only
Upload only the new training code ZIP. Do not regenerate or replace the verified dataset. Set CODE_ZIP if you saved it elsewhere. Extraction refuses existing destinations and unsafe members. The data itself is read directly from Drive; only code is copied into /content.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import json, shutil, stat, zipfile, uuid
from datetime import datetime, timezone
CODE_ZIP = Path('/content/drive/MyDrive/PP1_V3_Training_Code.zip')
CODE_ROOT = Path('/content/PP1_V3_training_code')
DATASET = Path('/content/drive/MyDrive/PP1_V3_runs/PP1_Synthetic_Motor_Dataset_V3_20261020T210904Z_b8c13141')
DRIVE_RUNS = Path('/content/drive/MyDrive/PP1_V3_training_runs')
TEST_REGISTRY = Path('/content/drive/MyDrive/PP1_V3_final_test_registry')
if CODE_ROOT.exists() or CODE_ROOT.is_symlink():
    raise FileExistsError('Code restore exists; use a fresh runtime without overwriting files')
with zipfile.ZipFile(CODE_ZIP) as archive:
    entries = archive.infolist(); seen = set()
    if len(entries)>1000 or sum(m.file_size for m in entries)>20_000_000:
        raise ValueError('Unexpected training code bundle size')
    for item in entries:
        name=item.filename; parts=name.split('/')
        if (not name.startswith('training/') or '\\' in name or ':' in name
                or any(p in ('','.','..') for p in parts) or any(ord(c)<32 for c in name)):
            raise ValueError('Unsafe ZIP path')
        if item.is_dir() or stat.S_IFMT(item.external_attr>>16) not in (0,stat.S_IFREG):
            raise ValueError('Only regular files are allowed')
        if name.casefold() in seen: raise ValueError('Duplicate ZIP name')
        seen.add(name.casefold())
    for name in seen:
        parts=name.split('/')
        if any('/'.join(parts[:i]) in seen for i in range(1,len(parts))):
            raise ValueError('File/directory conflict')
    assert archive.testzip() is None, 'ZIP CRC failure'
    CODE_ROOT.mkdir(exist_ok=False)
    for item in entries:
        target=CODE_ROOT.joinpath(*item.filename.split('/'))
        target.parent.mkdir(parents=True,exist_ok=True)
        with archive.open(item) as source, target.open('xb') as output:
            shutil.copyfileobj(source,output)
PACKAGE = CODE_ROOT/'training'
sys.path.insert(0,str(PACKAGE/'src'))
import pp1_training as training
training.require_colab()
training.verify_package()
assert DATASET.is_dir(), 'Verified Drive dataset not found; do not generate a replacement'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8]
RUN = Path('/content')/('PP1_V3_training_'+RUN_ID)
print('Dataset (read-only):',DATASET)
print('New output directory:',RUN)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset (read-only): /content/drive/MyDrive/PP1_V3_runs/PP1_Synthetic_Motor_Dataset_V3_20261020T210904Z_b8c13141
New output directory: /content/PP1_V3_training_20261020T214151Z_961db068


## 3. Dataset preflight (Colab only; no fitting)
Requires passing COMPLETE/integrity markers, 240 unique sessions, exact frozen split membership, expected engine/config/code hashes, actual schemas and class counts. Hashes all raw files but performs no test predictions. Any mismatch stops the workflow. Test class support is an audit disclosure, not tuning feedback.


In [3]:
preflight_report = training.preflight(DATASET, RUN)
print(json.dumps(preflight_report,indent=2))


{
  "status": "passed",
  "dataset_root": "/content/drive/MyDrive/PP1_V3_runs/PP1_Synthetic_Motor_Dataset_V3_20261020T210904Z_b8c13141",
  "sessions": 240,
  "split_counts": {
    "train": 168,
    "validation": 36,
    "test": 36
  },
  "class_counts": {
    "train": {
      "TS": [
        100,
        24,
        20,
        16,
        8
      ],
      "BS": [
        39,
        55,
        33,
        24,
        17
      ],
      "RS": [
        78,
        33,
        22,
        23,
        12
      ],
      "GI": [
        61,
        42,
        27,
        18,
        20
      ]
    },
    "validation": {
      "TS": [
        24,
        3,
        5,
        2,
        2
      ],
      "BS": [
        7,
        13,
        9,
        3,
        4
      ],
      "RS": [
        12,
        8,
        7,
        5,
        4
      ],
      "GI": [
        12,
        9,
        3,
        9,
        3
      ]
    },
    "test": {
      "TS": [
        18,
        10,
     

## 4. Review feature/target mapping and frozen search spaces
Actual V3 targets are TS, BS, RS, GI, and demo_MPS. Only the nine domain-specific allowlisted numerical features enter models. IDs are join keys. All four recordings share one split. Three candidate configurations per algorithm are frozen before fitting. Within-domain folds are identical for all algorithms.


In [4]:
print((RUN/'mapping.json').read_text())
print((RUN/'search_spaces.json').read_text())
print((PACKAGE/'configs/frozen_evaluation_protocol.json').read_text())


{
  "features": {
    "TS": [
      "TS_tremor_rms_ms2"
    ],
    "BS": [
      "BS_tap_count",
      "BS_tap_rate_per_min",
      "BS_tap_interval_std_s"
    ],
    "RS": [
      "RS_mean_gyro_peak_rad_s",
      "RS_movement_peaks"
    ],
    "GI": [
      "GI_step_candidates",
      "GI_cadence_steps_min",
      "GI_step_interval_std_s"
    ]
  },
  "actual_targets": {
    "TS": "TS",
    "BS": "BS",
    "RS": "RS",
    "GI": "GI"
  },
  "display_targets": {
    "TS": "TS_demo",
    "BS": "BS_demo",
    "RS": "RS_active_mobility_proxy_demo",
    "GI": "GI_demo"
  },
  "MPS": "sum(TS,BS,RS,GI)/16"
}

{
  "seed": 20261021,
  "threads": 1,
  "bootstrap_replicates": 2000,
  "default_configuration_index": 0,
  "simplicity_order": "Configuration index ascending; fixed before execution",
  "algorithm_tie_order": [
    "RF",
    "SVM",
    "XGB",
    "CatBoost"
  ],
  "spaces": {
    "RF": [
      {
        "n_estimators": 120,
        "max_depth": 3,
        "min_samples_leaf": 3,
        

## 5. Explicit training approval gate
Leave FALSE until the user explicitly approves training. This gate also controls validation inference. No test inference is authorized by this switch.

In [5]:
APPROVE_TRAINING = True
print('Training approved:', APPROVE_TRAINING)


Training approved: True


## 6. Training-only CV and 16 final candidate fits
No validation or test observations enter fitting. Each fold fits its own imputer/scaler. Hyperparameters use mean fold MAE, then mean macro F1, then the prespecified simpler configuration index. Rare-class policy can reduce folds or skip search; it never changes labels. Every selected candidate is fit on the 168 training sessions only. No post-validation refit.


In [6]:
if APPROVE_TRAINING:
    cv_results = training.train(RUN, approved=True)
    print(json.dumps(cv_results,indent=2))
else:
    print('Training disabled; no model fitting performed.')


Fitted TS RF
Fitted TS SVM
Fitted TS XGB
Fitted TS CatBoost
Fitted BS RF
Fitted BS SVM
Fitted BS XGB
Fitted BS CatBoost
Fitted RS RF
Fitted RS SVM
Fitted RS XGB
Fitted RS CatBoost
Fitted GI RF
Fitted GI SVM
Fitted GI XGB
Fitted GI CatBoost
[
  {
    "domain": "TS",
    "algorithm": "RF",
    "configuration_index": 1,
    "folds": 3,
    "CV_MAE": 0.011904761904761904,
    "CV_macro_f1": 0.9855107569242776
  },
  {
    "domain": "TS",
    "algorithm": "SVM",
    "configuration_index": 2,
    "folds": 3,
    "CV_MAE": 0.011904761904761904,
    "CV_macro_f1": 0.9855107569242776
  },
  {
    "domain": "TS",
    "algorithm": "XGB",
    "configuration_index": 0,
    "folds": 3,
    "CV_MAE": 0.023809523809523808,
    "CV_macro_f1": 0.9543996458131665
  },
  {
    "domain": "TS",
    "algorithm": "CatBoost",
    "configuration_index": 0,
    "folds": 3,
    "CV_MAE": 0.011904761904761904,
    "CV_macro_f1": 0.9855107569242776
  },
  {
    "domain": "BS",
    "algorithm": "RF",
    "configurat

## 7. Validation comparisons and domain selection
Select by validation MAE, then macro F1, then RF/SVM/XGB/CatBoost tie order. Search spaces cannot expand. Inspect 5x5 confusion matrices, class support, Wilson intervals and session-bootstrap MAE intervals. Only 36 sessions: uncertainty is conditional and rare-class estimates are unreliable. The rule engine is a consistency baseline that defines the targets.


In [7]:
if APPROVE_TRAINING:
    selection = training.validate(RUN, approved=True)
    print(json.dumps(selection,indent=2))
    print((RUN/'validation/comparison.csv').read_text())
else:
    print('Validation inference disabled.')


{
  "TS": "SVM",
  "BS": "RF",
  "RS": "RF",
  "GI": "RF"
}
domain,algorithm,n,exact_accuracy,within_1_accuracy,macro_f1,balanced_accuracy,MAE,class_support,uncertainty
TS,RF,36,0.9444444444444444,1.0,0.86,0.8333333333333334,0.05555555555555555,"[24, 3, 5, 2, 2]","{""exact_accuracy_wilson_95"": [0.8185534279135275, 0.9846300133358382], ""within_1_accuracy_wilson_95"": [0.9035813714055363, 1.0], ""MAE_session_bootstrap_95"": [0.0, 0.1388888888888889], ""class_recall_wilson_95"": [[0.8620237953250198, 1.0], [0.2076596008020477, 0.9385080552796037], [0.5655175352168251, 1.0], [0.09453120573423074, 0.9054687942657693], [0.34238022750665303, 1.0]], ""limitation"": ""Conditional on these simulator sessions and frozen models; not source/clinical uncertainty. Rare-class intervals are unstable; no multiplicity correction.""}"
TS,SVM,36,0.9722222222222222,1.0,0.9492063492063492,0.96,0.027777777777777776,"[24, 3, 5, 2, 2]","{""exact_accuracy_wilson_95"": [0.8583028134672614, 0.9950795928601897], 

## 8. Freeze before test and back up frozen checkpoint
Hashes all 16 candidate artifacts, feature mapping, code/configuration, CV/validation reports and selection. No model refitting occurs. A fresh Drive checkpoint is saved before test. Existing output paths are refused.


In [8]:
if APPROVE_TRAINING:
    training.freeze(RUN)
    FROZEN_BACKUP = DRIVE_RUNS/(RUN.name+'_frozen')
    training.backup(RUN,FROZEN_BACKUP)
    print('Frozen checkpoint:',FROZEN_BACKUP)
else:
    print('No trained checkpoint to freeze.')


Frozen checkpoint: /content/drive/MyDrive/PP1_V3_training_runs/PP1_V3_training_20261020T214151Z_961db068_frozen


## 9. Separate final-test approval gate
Leave FALSE until explicit approval after reviewing the frozen selection. Evaluation claims a persistent dataset-specific marker in Drive BEFORE any test predictions. Repeated calls, including from a new training run, are blocked. An interrupted test attempt stays claimed; do not delete its marker or silently retry.


In [9]:
APPROVE_FINAL_TEST = True
print('Final test approved:', APPROVE_FINAL_TEST)


Final test approved: True


## 10. Single frozen final evaluation
Evaluates all 16 frozen candidates, the selected four-model system and unchanged deterministic rule baseline. No selection changes follow test results. Reports exact/within-1 accuracy, fixed-five-class macro F1, balanced accuracy, MAE, confusion matrices, support, uncertainty, per-session scores and MPS. MPS is calculated from scores; no MPS regressor exists.


In [10]:
if APPROVE_FINAL_TEST:
    assert (RUN/'FROZEN.json').is_file(), 'Freeze models first'
    assert FROZEN_BACKUP.is_dir(), 'Back up frozen checkpoint first'
    final_report = training.final_test(RUN,TEST_REGISTRY,approved=True)
    print(json.dumps(final_report,indent=2))
else:
    print('Final test disabled; no test model predictions performed.')


{
  "sessions": 36,
  "complete_sessions": 36,
  "complete_session_coverage": 1.0,
  "MPS_MAE": 0.005208333333333333,
  "MPS_MAE_session_bootstrap_95": [
    0.0,
    0.012152777777777778
  ],
  "interpretation": "Demo index agreement only; conditional simulator-session uncertainty"
}


## 11. Back up final models/reports to Google Drive
Copies the versioned run to a new destination and verifies file hashes. The dataset, original engine, earlier checkpoints, and frozen Drive backup are never overwritten. Final-test markers remain separate and persistent.


In [11]:
if (RUN/'test/TEST_COMPLETE.json').is_file():
    FINAL_BACKUP = DRIVE_RUNS/(RUN.name+'_final')
    print(training.backup(RUN,FINAL_BACKUP))
else:
    print('No completed final test to back up. Preserve any frozen checkpoint and failed-test marker.')


/content/drive/MyDrive/PP1_V3_training_runs/PP1_V3_training_20261020T214151Z_961db068_final


## Stop and scientific interpretation
No result establishes clinical accuracy, real-source independence or Parkinson probability. Models imitate the unchanged PP1 rules on the same simulator. Do not revise the dataset, hyperparameters or selected algorithms using test results. Future real-device operation must report extracted features without validated 0-4 scores or MPS. No real-device scoring API is introduced by this notebook.


In [12]:

from pathlib import Path
import pandas as pd

test_dir = RUN / "test"
assert test_dir.is_dir(), "Final test directory missing!"

print("FINAL TEST FILES:")
for file in sorted(test_dir.rglob("*")):
    if file.is_file():
        print(file.relative_to(test_dir))

print("\nFINAL TEST ACCURACY TABLES:")

found = False

for file in sorted(test_dir.rglob("*.csv")):
    df = pd.read_csv(file)

    if "exact_accuracy" in df.columns:
        found = True
        print("\nFILE:", file.name)

        columns = [
            c for c in [
                "domain",
                "algorithm",
                "n",
                "exact_accuracy",
                "within_1_accuracy",
                "macro_f1",
                "MAE"
            ]
            if c in df.columns
        ]

        print(df[columns].to_string(index=False))

if not found:
    print("No accuracy CSV detected; inspect listed files.")


FINAL TEST FILES:
BS/CatBoost/confusion_matrix.csv
BS/CatBoost/metrics.json
BS/CatBoost/predictions.csv
BS/RF/confusion_matrix.csv
BS/RF/metrics.json
BS/RF/predictions.csv
BS/SVM/confusion_matrix.csv
BS/SVM/metrics.json
BS/SVM/predictions.csv
BS/XGB/confusion_matrix.csv
BS/XGB/metrics.json
BS/XGB/predictions.csv
GI/CatBoost/confusion_matrix.csv
GI/CatBoost/metrics.json
GI/CatBoost/predictions.csv
GI/RF/confusion_matrix.csv
GI/RF/metrics.json
GI/RF/predictions.csv
GI/SVM/confusion_matrix.csv
GI/SVM/metrics.json
GI/SVM/predictions.csv
GI/XGB/confusion_matrix.csv
GI/XGB/metrics.json
GI/XGB/predictions.csv
RS/CatBoost/confusion_matrix.csv
RS/CatBoost/metrics.json
RS/CatBoost/predictions.csv
RS/RF/confusion_matrix.csv
RS/RF/metrics.json
RS/RF/predictions.csv
RS/SVM/confusion_matrix.csv
RS/SVM/metrics.json
RS/SVM/predictions.csv
RS/XGB/confusion_matrix.csv
RS/XGB/metrics.json
RS/XGB/predictions.csv
TEST_COMPLETE.json
TS/CatBoost/confusion_matrix.csv
TS/CatBoost/metrics.json
TS/CatBoost/predi

In [13]:

import json

report_path = RUN / "test" / "selected_system.json"

assert report_path.is_file(), "Selected system report missing!"

with report_path.open() as f:
    report = json.load(f)

print(json.dumps(report, indent=2))


{
  "sessions": 36,
  "complete_sessions": 36,
  "complete_session_coverage": 1.0,
  "MPS_MAE": 0.005208333333333333,
  "MPS_MAE_session_bootstrap_95": [
    0.0,
    0.012152777777777778
  ],
  "interpretation": "Demo index agreement only; conditional simulator-session uncertainty"
}


In [14]:

import pandas as pd

file = RUN / "test" / "selected_system_predictions.csv"

assert file.is_file(), "Selected predictions file missing!"

df = pd.read_csv(file)

print("Rows:", len(df))
print("\nCOLUMN NAMES:")
print(df.columns.tolist())

print("\nFIRST 3 SYNTHETIC SESSIONS:")
print(df.head(3).to_string(index=False))


Rows: 36

COLUMN NAMES:
['session_id', 'TS_predicted', 'BS_predicted', 'RS_predicted', 'GI_predicted', 'TS_reference', 'BS_reference', 'RS_reference', 'GI_reference', 'predicted_demo_MPS', 'reference_demo_MPS', 'complete']

FIRST 3 SYNTHETIC SESSIONS:
               session_id  TS_predicted  BS_predicted  RS_predicted  GI_predicted  TS_reference  BS_reference  RS_reference  GI_reference  predicted_demo_MPS  reference_demo_MPS  complete
V3_MAIN_healthy_like_0001             0             2             0             0             0             2             0             0              0.1250              0.1250      True
V3_MAIN_healthy_like_0002             0             2             0             0             1             2             0             0              0.1250              0.1875      True
V3_MAIN_healthy_like_0006             0             2             0             1             0             2             0             1              0.1875              0.1875      T

In [15]:

import pandas as pd
import numpy as np

df = pd.read_csv(
    RUN / "test" / "selected_system_predictions.csv"
)

domains = ["TS", "BS", "RS", "GI"]

complete = df[df["complete"].astype(str).str.lower() == "true"].copy()

correct = pd.DataFrame({
    d: complete[f"{d}_predicted"] == complete[f"{d}_reference"]
    for d in domains
})

complete["all_four_correct"] = correct.all(axis=1)

joint_accuracy = complete["all_four_correct"].mean() * 100

print("FINAL SELECTED SYSTEM RESULTS")
print("Complete sessions:", len(complete))
print("All four correct:",
      int(complete["all_four_correct"].sum()))
print(f"Four-score joint exact accuracy: {joint_accuracy:.2f}%")

print("\nDOMAIN EXACT ACCURACY")
for d in domains:
    print(f"{d}: {correct[d].mean() * 100:.2f}%")

mps_mae = np.mean(
    np.abs(
        complete["predicted_demo_MPS"] -
        complete["reference_demo_MPS"]
    )
)
print(f"\nMPS MAE: {mps_mae:.6f}")

print("\nSESSIONS WITH INCORRECT SCORES")
errors = complete.loc[
    ~complete["all_four_correct"],
    ["session_id"] +
    [f"{d}_predicted" for d in domains] +
    [f"{d}_reference" for d in domains]
]
print(errors.to_string(index=False))


FINAL SELECTED SYSTEM RESULTS
Complete sessions: 36
All four correct: 33
Four-score joint exact accuracy: 91.67%

DOMAIN EXACT ACCURACY
TS: 94.44%
BS: 100.00%
RS: 100.00%
GI: 97.22%

MPS MAE: 0.005208

SESSIONS WITH INCORRECT SCORES
               session_id  TS_predicted  BS_predicted  RS_predicted  GI_predicted  TS_reference  BS_reference  RS_reference  GI_reference
V3_MAIN_healthy_like_0002             0             2             0             0             1             2             0             0
V3_MAIN_healthy_like_0014             0             1             0             0             1             1             0             0
V3_MAIN_symptom_like_0049             1             2             1             1             1             2             1             0
